# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-') and len(api_key) > 10:
    print('OpenAI API key found')
else:
    print('OpenAI API key missing or malformed; check your .env file')

openai_client = OpenAI()
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

OpenAI API key found


In [10]:
# here is the question

question = """
Compare training a smaller LLM using (a) distillation from a model you control versus (b) training on outputs collected from a third-party hosted model via API. What’s similar technically, and what differs in terms of risk (quality, evaluation leakage, dependence on teacher quirks, policy/ToS considerations)?
"""

In [12]:
# Get gpt-4o-mini to answer, with streaming

gpt_stream = openai_client.chat.completions.create(
    model=MODEL_GPT,
    messages=[
        {
            'role': 'system',
            'content': 'You are a concise and helpful technical tutor. Explain clearly and include a short example when useful.'
        },
        {'role': 'user', 'content': question}
    ],
    stream=True
)

gpt_response = ''
display_handle = display(Markdown(''), display_id=True)

for chunk in gpt_stream:
    delta = chunk.choices[0].delta.content or ''
    gpt_response += delta
    update_display(Markdown(gpt_response), display_id=display_handle.display_id)

When training a smaller LLM (Large Language Model), both distillation from a model you control and training on outputs collected from a third-party hosted model via API are viable approaches, but they differ technically and in terms of risks.

### Similarities:
1. **Knowledge Transfer**: Both methods aim to transfer knowledge from a larger model to a smaller one. In distillation, this is direct through logits/outputs; in the second approach, it’s indirect from generated text.
2. **Fine-tuning**: In both cases, you may fine-tune the smaller model on the respective data (distillation outputs or API outputs) to improve performance on your specific tasks.

### Differences:

#### Technical Differences:
- **Distillation (a)**: 
  - Involves directly accessing a previously trained large model that you control.
  - You have the ability to extract model logits (i.e., the probabilities over the vocabulary) which guide the smaller model more effectively on what the original model thought.
  - You retain control over the training parameters, adjustments, and methodologies.

- **API Outputs (b)**: 
  - Utilizes generated outputs from a third-party model, which may be less consistent or unpredictable.
  - You don’t have access to the inner workings or logits, meaning the smaller model learns from the completed outputs rather than from specific model behaviors.
  - You are limited in terms of the data quality and may need to deal with varying output styles or biases from the third-party model.

#### Risk Considerations:
- **Quality**:
  - **Distillation (a)** allows for a reliable source of high-quality data since the model is controlled and can be evaluated for performance.
  - **API Outputs (b)** can introduce variability and quality issues due to potential inconsistencies or restrictions in the API's outputs.

- **Evaluation Leakage**:
  - With distillation, since you control the training process, you can ensure some level of separation from direct output checks, minimizing leakage risks.
  - Training on API outputs may inadvertently introduce evaluation leakage if the third-party model inadvertently exposes known data or patterns.

- **Dependence on Teacher Quirks**:
  - **Distillation (a)** allows you to be aware of and mitigate the quirks of the teacher model, using its tuning to balance behaviors in the smaller model.
  - In the case of **API Outputs (b)**, you inherit unknown quirks and biases which can negatively impact the smaller model's performance and behavior.

- **Policy/ToS Considerations**:
  - When distilling your model (a), you can create your data and are less subject to external terms of service.
  - Training based on a third-party hosted model (b) may infringe on the API's Terms of Service, particularly regarding redistribution or usage of generated outputs, leading to potential legal risks.

### Example:
- **Distillation**: You train a smaller version of GPT-3 by distilling it from the full model, generating logits during its training phase to guide the smaller model, ensuring robust performance in predetermined tasks.

- **Training on API outputs**: You collect thousands of text completions from a third-party LLM via an API and use these outputs to train a smaller model, but you may discover that the stylistic and factual accuracy of those outputs varies widely, affecting the smaller model's robustness.

In summary, while both methods aim to leverage the capabilities of larger LLMs for the benefit of a smaller model, distillation generally provides greater consistency, control, and alignment to the intended model behavior, while API training introduces potential quality issues and legal concerns.

In [13]:
# Get Llama 3.2 to answer, with streaming

llama_stream = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {
            'role': 'system',
            'content': 'You are a concise and helpful technical tutor. Explain clearly and include a short example when useful.'
        },
        {'role': 'user', 'content': question}
    ],
    stream=True
)

llama_text = ''
display_handle = display(Markdown(''), display_id=True)

for chunk in llama_stream:
    delta = chunk.choices[0].delta.content or ''
    llama_text += delta
    update_display(Markdown(llama_text), display_id=display_handle.display_id)


Distilling a smaller LLM using a controlled model involves modifying the original model's architecture to mimic its training data distribution, typically by reducing computational resources or modifying the model's complexity. This approach allows for direct control over the output quality, enabling evaluation and fine-tuning.

On the other hand, training on outputs collected from a third-party hosted model via API involves using pre-trained models from a provider, which may not be directly under your control. Here's a comparison of similar and differing technical aspects:

**Similarities:**

1. **Use of Distillation**: Both approaches use distillation techniques to adapt a smaller LLM.
2. **Learning from Existing Models**: Both involve leveraging pre-trained models for efficient knowledge transfer.

**Differences in terms of risk:**

1. **Quality Concerns**:
   - (a) By training on the original model, you have direct control over the quality of the distilled model and can ensure consistency in output distribution.
   - (b) Using a third-party hosted model introduces uncertainty regarding the quality of the collected outputs, which may not match the expected results.

2. **Evaluation Leakage**:
   - (a) With full control, it's challenging to introduce leakage by intentionally distorting evaluation metrics or biases.
   - (b) As an external user, you're more vulnerable to potential evaluation metric manipulation or data drift used in distillation techniques by the third-party provider.

3. **Dependence on Teacher Quirks**:
   - (a) Controlling your own model means minimizing dependence on teacher quirks and focus can be achieved through careful experimentation.
   - (b) Using a third-party hosted model limits your ability to directly influence or correct its behaviors, increasing the risk of being tied to specific quirks.

4. **Policy/ToS Considerations**:
   - (a) By using a controlled model, you follow data and model usage policies by the original provider.
   - (b) As an external user of third-party hosted models, you're bound by terms of service that might be different from what you'd prefer.

Example to illustrate distillation in both scenarios:

**Scenario A:** Training on a smaller LLM using distillation

```python
import torch
from torch.nn import Linear

# Assume "original_model" is the pre-trained model with architecture known beforehand
class DistillinNetwork(torch.nn.Module):
    def __init__(self, original_layer, n_class):
        super(DistillinNetwork, self).__init__()
        self.originalLayer = original_layer
        self.fullyConnect = Linear(original_layer.in_features, n_class)

    def forward(self, x):
        return self.fullyConnect(x)
```

**Scenario B:** Training on a third-party hosted model via API

```python
import requests

# Assume "third_party_model" is the pre-trained model provided by an external client
class HostedDistiller(torch.nn.Module):
    def __init__(self, n_class):
        super(HostedDistiller, self).__init__()
        # Assume 'n_class' input features known beforehand
        # Assume third party provides output features via API

    def forward(self, x): 
        response = requests.post("https://external_service.com/api", json={"data": [x]})
        collected_data = response.json()
        return torch.tensor(collected_data["output"])
```

In conclusion, controlling your own model offers more flexibility and reliability in terms of quality consistency but imposes policy compliance on the original model creator. Working on third_party hosted models introduces uncertainty about output quality while offering access to a diverse pre-trained base that can be fine-tuned with distillation.